# Python 200: Assignment 04 - Warmup Exercises

In [7]:
import os
os.makedirs("outputs", exist_ok=True)

In [8]:
import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
from pathlib import Path

# standard device check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version:     {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")

Using device: cuda
PyTorch version:     2.10.0+cu128
TorchVision version: 0.25.0+cu128


In [9]:


# --- Q1: Creating and Inspecting Tensors ---
print("--- Tensor Q1 Output ---")

a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
b = torch.zeros(2, 3)
c = torch.ones(4)

# quick list
tensors = [("a", a), ("b", b), ("c", c)]

for name, t in tensors:
    print(f"Tensor '{name}':")
    print(f"  Value:\n{t}")
    print(f"  Shape:  {t.shape}")
    print(f"  Dtype:  {t.dtype}")
    print(f"  Device: {t.device}\n")



--- Tensor Q1 Output ---
Tensor 'a':
  Value:
tensor([[1., 2., 3.],
        [4., 5., 6.]])
  Shape:  torch.Size([2, 3])
  Dtype:  torch.float32
  Device: cpu

Tensor 'b':
  Value:
tensor([[0., 0., 0.],
        [0., 0., 0.]])
  Shape:  torch.Size([2, 3])
  Dtype:  torch.float32
  Device: cpu

Tensor 'c':
  Value:
tensor([1., 1., 1., 1.])
  Shape:  torch.Size([4])
  Dtype:  torch.float32
  Device: cpu



In [10]:
# --- Q2: Tensor Math Operations ---
print("\n--- Tensor Q2 Output ---")
x = torch.tensor([1.0, 4.0, 9.0, 16.0, 25.0])

print(f"Square root: {torch.sqrt(x)}")
print(f"Sum:         {x.sum()}")
print(f"Mean:        {x.mean()}")
print(f"Argmax:      {x.argmax()}")

"""
COMMENT: What does .argmax() give you in a classifier?
In a classifier, a neural network outputs list of probabilities (or raw scores 
called "logits") for every possible class. For 1,000 classes, it outputs 
a tensor of 1,000 numbers. Calling .argmax() returns the *index* of the highest 
number in that tensor. That index corresponds exactly to the network's final 
prediction (e.g., index 284 might map to the class label "cat").
"""

# --- Q3: Moving Tensors between CPU and GPU ---
print("\n--- Tensor Q3 Output ---")

a_gpu   = a.to(device)
print(f"a_gpu device: {a_gpu.device}")

a_back  = a_gpu.cpu()
a_numpy = a_back.numpy()
print(f"numpy type: {type(a_numpy)}")
print(f"numpy values:\n{a_numpy}")

"""
COMMENT: Why does PyTorch require .cpu() before .numpy()?
NumPy is fundamentally a CPU-bound library. It is designed to read and manipulate 
data stored in system RAM. It cannot  
read data sitting in Graphics Card's VRAM. Have to explicitly 
move the tensor back across the hardware bridge to the CPU before NumPy can touch it.
"""

# --- Q4: Shape Manipulation ---
print("\n--- Tensor Q4 Output ---")
t = torch.arange(24).float()

t_4_6 = t.reshape(4, 6)
print(f"Reshaped to (4, 6):   {t_4_6.shape}")

t_2_3_4 = t.reshape(2, 3, 4)
print(f"Reshaped to (2, 3, 4): {t_2_3_4.shape}")

# Adding a new dimension at position 0 (often called "unsqueezing")
t_batched = t_4_6.unsqueeze(0) 
print(f"Added dimension at 0:  {t_batched.shape}")

"""
COMMENT: What operation accomplishes this and why does it matter?
The operation is called "unsqueezing" (using .unsqueeze(0)). 
It matters because neural networks are strictly designed to process data in 
"batches" to maximize parallel processing. Even if you are just passing a single 
image to the model for prediction, the model will crash if it doesn't see that 
batch dimension. You have to unsqueeze it to create a "batch of 1".
"""

# --- Q5: Matrix Multiplication ---
print("\n--- Tensor Q5 Output ---")
np_a = np.array([[1.0, 2.0], [3.0, 4.0]])
np_b = np.array([[5.0, 6.0], [7.0, 8.0]])
t_a  = torch.tensor(np_a, dtype=torch.float32)
t_b  = torch.tensor(np_b, dtype=torch.float32)

# NumPy matrix multiplication (can use np.matmul() or the @ operator)
np_result = np_a @ np_b
print(f"NumPy Result:\n{np_result}")

# PyTorch matrix multiplication (can use torch.matmul() or the @ operator)
t_result = t_a @ t_b
print(f"\nPyTorch Result:\n{t_result}")

"""
COMMENT: What role does matrix multiplication play in a neural network layer?
Matrix multiplication is the core engine of a neural network. When data 
passes through a layer, the input features (like pixel values) are matrix-multiplied 
by the layer's "weights" (the parameters the network is learning). This multiplication 
is how the network combines individual input signals to form more complex patterns 
and representations in the next layer.
"""


--- Tensor Q2 Output ---
Square root: tensor([1., 2., 3., 4., 5.])
Sum:         55.0
Mean:        11.0
Argmax:      4

--- Tensor Q3 Output ---
a_gpu device: cuda:0
numpy type: <class 'numpy.ndarray'>
numpy values:
[[1. 2. 3.]
 [4. 5. 6.]]

--- Tensor Q4 Output ---
Reshaped to (4, 6):   torch.Size([4, 6])
Reshaped to (2, 3, 4): torch.Size([2, 3, 4])
Added dimension at 0:  torch.Size([1, 4, 6])

--- Tensor Q5 Output ---
NumPy Result:
[[19. 22.]
 [43. 50.]]

PyTorch Result:
tensor([[19., 22.],
        [43., 50.]])


'\nCOMMENT: What role does matrix multiplication play in a neural network layer?\nMatrix multiplication is the core engine of a neural network. When data \npasses through a layer, the input features (like pixel values) are matrix-multiplied \nby the layer\'s "weights" (the parameters the network is learning). This multiplication \nis how the network combines individual input signals to form more complex patterns \nand representations in the next layer.\n'

In [11]:

# Pretrained Models


# --- Q1: Loading ResNet18 ---
print("--- Model Q1 Output ---")

weights = ResNet18_Weights.DEFAULT
model   = models.resnet18(weights=weights)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

"""
COMMENT: What does this tell you about the practical value of pretrained weights?
Pretrained weights saves massive amounts of time and compute budget. 
Instead of using up loads of GPU capacity , you can just 
download those foundational visual concepts instantly.
"""


# --- Q2: Model Architecture ---
print("\n--- Model Q2 Output ---")

# print(model) 

"""
COMMENT: 
1. What is the name of the final layer, and its output size?
The final layer is named 'fc' (which stands for Fully Connected layer, or Linear layer). 
Its output size is `out_features=1000`, corresponding to the 1,000 different 
categories in the ImageNet dataset.

2. What does it mean for a network to be "deep"?
A "deep" network has multiple, sequential layers stacked on top of each other. 
The earliest layers (like layer1) learn simple things like basic edges and colors. 
The deeper layers (layer3, layer4) combine those simple shapes into complex, 
high-level concepts (like "a dog's ear" or "a car tire"). "Deep" just means a 
long hierarchy of pattern recognition!
"""


# --- Q3: GPU and Evaluation Mode ---
print("\n--- Model Q3 Output ---")

model = model.to(device)
model.eval()
print("Model ready for inference.")

"""
COMMENT:
1. What does .to(device) do?
It physically moves the 11 million weights/parameters of the model from your CPU's 
RAM into the GPU's VRAM. PyTorch requires the model and the input images to be on 
the exact same device to perform matrix math, otherwise it will crash.

2. What does model.eval() do? Name a layer type that changes behavior.
It switches the model from "training" mode to "evaluation" (or inference) mode. 
Some layers behave differently when learning vs. when predicting. For example, 
a 'Dropout' layer randomly turns off neurons during training to prevent overfitting, 
but during .eval(), it leaves all neurons turned on so you get a consistent, 
deterministic prediction. (Batch Normalization also locks its statistics in eval mode).
"""


# --- Q4: Preprocessing Transforms ---
print("\n--- Model Q4 Output ---")
preprocess = weights.transforms()
print(preprocess)

"""
COMMENT: What does each step do?
1. Resize/Crop: Neural networks require every single input image to be the exact 
   same shape (in this case, 224x224 pixels) so the math aligns properly. This step 
   standardizes the size regardless of the original image dimensions.
   
2. ToTensor(): This does two things. It converts the image from a PIL Image to a 
   PyTorch Tensor (moving from HeightxWidthxChannels to ChannelsxHeightxWidth). It 
   also squashes the raw pixel values (which are normally 0 to 255) down to a 
   float range between 0.0 and 1.0.
   
3. Normalization (Mean/Std): Normalization centers the data around zero, which 
   helps the math run faster and more stably. We use ImageNet's highly specific 
   mean and std (rather than a generic 0.5) because this specific model was trained 
   on data that looked exactly like that. If we change the normalization math now, 
   the model will be confused because the "colors" will look completely different 
   than what it learned!
"""

--- Model Q1 Output ---
Total parameters:     11,689,512
Trainable parameters: 11,689,512

--- Model Q2 Output ---

--- Model Q3 Output ---
Model ready for inference.

--- Model Q4 Output ---
ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


'\nCOMMENT: What does each step do?\n1. Resize/Crop: Neural networks require every single input image to be the exact \n   same shape (in this case, 224x224 pixels) so the math aligns properly. This step \n   standardizes the size regardless of the original image dimensions.\n   \n2. ToTensor(): This does two things. It converts the image from a PIL Image to a \n   PyTorch Tensor (moving from HeightxWidthxChannels to ChannelsxHeightxWidth). It \n   also squashes the raw pixel values (which are normally 0 to 255) down to a \n   float range between 0.0 and 1.0.\n   \n3. Normalization (Mean/Std): Normalization centers the data around zero, which \n   helps the math run faster and more stably. We use ImageNet\'s highly specific \n   mean and std (rather than a generic 0.5) because this specific model was trained \n   on data that looked exactly like that. If we change the normalization math now, \n   the model will be confused because the "colors" will look completely different \n   than w

In [12]:
# 
# Running Inference
# 
import random
import torch.nn.functional as F

print("\n--- Inference Setup ---")
random.seed(42)
DATA_DIR = Path("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test")
LABELS   = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

def load_sample_image(label):
    """Load a random image file from the given class folder."""
    class_dir = DATA_DIR / label
    img_path  = random.choice(list(class_dir.glob("*.jpg")))
    return Image.open(img_path).convert("RGB"), img_path.name

imagenet_classes = weights.meta["categories"]
print(f"Number of classes: {len(imagenet_classes)}")
print(f"First 5 labels: {imagenet_classes[:5]}")


# --- Q1: Inference Function ---
print("\n--- Inference Q1 Output ---")

def get_top5_predictions(model, preprocess, image, device, class_labels):
    """
    Run inference on a PIL image and return the top-5 predictions.
    
    """
    # Step 1: Preprocess and add batch dimension
    input_tensor = preprocess(image).unsqueeze(0).to(device)
    
    # Step 2: Run inference without calculating gradients
    with torch.no_grad():
        output = model(input_tensor)
        
    # Step 3: Convert logits to probabilities
    probs = F.softmax(output[0], dim=0)
    
    # Step 4: Get top 5
    top_probs, top_indices = torch.topk(probs, 5)
    
    # Step 5: Build list
    preds = []
    for i in range(5):
        class_name = class_labels[top_indices[i].item()]
        probability = top_probs[i].item()
        preds.append((class_name, probability))
        
    return preds

# Test on a mountain image
img_m, img_name_m = load_sample_image("mountain")
preds_m = get_top5_predictions(model, preprocess, img_m, device, imagenet_classes)

print(f"Top-5 predictions for '{img_name_m}':")
for class_name, prob in preds_m:
    print(f"  {class_name:30s}  {prob:.4f}")

"""
COMMENT: Does the top prediction make sense? 
Yes. ImageNet doesn't just have a generic "mountain" category; it has 
highly specific categories like "alp", "valley", and "volcano". The model's top 
predictions map onto the visual features of a mountain scene, even if 
the exact string "mountain" isn't the number one result.
"""


# --- Q2: Loop Over All Classes ---
print("\n--- Inference Q2 Output ---")

for label in LABELS:
    img, img_name = load_sample_image(label)
    preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)[:3]
    print(f"\n[{label}]  {img_name}")
    for class_name, prob in preds:
        print(f"  {class_name:30s}  {prob:.4f}")

"""
COMMENT: Confidence patterns
The model is confident (high top-1 prob) on classes with very distinct 
textures or objects, like "forest" (predicting "valley" or "forest") or "street" 
(predicting specific vehicles or road features). It isleast confident on 
"sea" or "glacier" because massive bodies of water, sky, and ice can look visually 
similar, causing the probabilities to split across categories like "promontory", 
"breakwater", or "lakeside".
"""


# --- Q3: Logits vs Probabilities ---
print("\n--- Inference Q3 Output ---")

img_f, _ = load_sample_image("forest")
input_tensor = preprocess(img_f).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)

probs = F.softmax(logits[0], dim=0)

print(f"Logit  range: min={logits.min():.2f}, max={logits.max():.2f}")
print(f"Prob   range: min={probs.min():.6f}, max={probs.max():.4f}")
print(f"Probs sum to: {probs.sum():.6f}")
print(f"Top prediction: {imagenet_classes[probs.argmax().item()]}  ({probs.max():.4f})")

"""
COMMENT: Why output logits? Which to use for filtering?
Neural networks output logits internally because they are unconstrained (they can 
be massive negative or positive numbers), which makes the calculus for updating 
weights during training much more numerically stable. 

In a production pipeline, you would work with probabilities. Because they are 
forced to sum to 1.0, you can set a logical, human-readable threshold (e.g., 
"flag any prediction under 0.85 for human review"). You can't do that with logits 
because a logit of 12.5 means nothing out of context.
"""


# --- Q4: Visualization ---
print("\n--- Inference Q4 Output ---")


img_v, img_name_v = load_sample_image("street")
preds_v = get_top5_predictions(model, preprocess, img_v, device, imagenet_classes)

# Extract
names = [p[0] for p in preds_v]
probs = [p[1] for p in preds_v]

# Create 1x2 grid
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot image left
axes[0].imshow(img_v)
axes[0].axis('off')
axes[0].set_title(f"Image: {img_name_v}")

# Plot the bar chart  right 
axes[1].barh(names[::-1], probs[::-1], color='mediumpurple')
axes[1].set_xlabel('Probability')
axes[1].set_title('Top 5 ImageNet Predictions')
axes[1].set_xlim(0, 1.0) # Lock x-axis from 0 to 1

plt.tight_layout()
plt.savefig('outputs/warmup_inference_viz.png')
plt.clf()

print("Saved outputs/warmup_inference_viz.png")

"""
COMMENT: Dashboard Adaptation
For a non-technical dashboard, this side-by-side view is good. To adapt it, I would color-code the bars: green if the top 
probability is > 0.80, yellow if it's 0.50-0.80, and red if it's < 0.50. A threshold 
of 0.70 or 0.80 is usually a solid starting point for deciding a model is "confident 
enough" to act automatically without human review.
"""


--- Inference Setup ---
Number of classes: 1000
First 5 labels: ['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead']

--- Inference Q1 Output ---
Top-5 predictions for '24204.jpg':
  alp                             0.4911
  volcano                         0.2076
  valley                          0.2016
  promontory                      0.0184
  mountain tent                   0.0169

--- Inference Q2 Output ---

[buildings]  24258.jpg
  palace                          0.4301
  gondola                         0.1305
  monastery                       0.0624

[forest]  23309.jpg
  viaduct                         0.3853
  totem pole                      0.1089
  cliff                           0.0419

[glacier]  20272.jpg
  volcano                         0.3854
  valley                          0.3297
  promontory                      0.1216

[mountain]  20662.jpg
  ski                             0.5933
  alp                             0.3821
  snowmobile            

'\nCOMMENT: Dashboard Adaptation\nFor a non-technical dashboard, this side-by-side view is good. To adapt it, I would color-code the bars: green if the top \nprobability is > 0.80, yellow if it\'s 0.50-0.80, and red if it\'s < 0.50. A threshold \nof 0.70 or 0.80 is usually a solid starting point for deciding a model is "confident \nenough" to act automatically without human review.\n'

<Figure size 1200x500 with 0 Axes>